# ⚠️ EJECUTAR EN KAGGLE (T4) — no ejecutar en local

# 05 — Entrenamiento QLoRA de SkillGen con **Unsloth** (experimento E1)

Este es el notebook de producción de la Fase 3 y es, desde el
2026-10-23, un notebook **de Unsloth de verdad**: el modelo se carga
con `FastLanguageModel.from_pretrained`, LoRA se aplica con
`FastLanguageModel.get_peft_model`, la inferencia de control usa
`FastLanguageModel.for_inference` y el export GGUF usa
`save_pretrained_gguf`. Unsloth es el motor de entrenamiento de
SkillGen por decisión explícita de Roger — no un adorno instalado en
una celda.

**Honestidad primero:** este notebook **no ha sido ejecutado** en la
VM local de desarrollo (no tiene GPU y Unsloth exige CUDA). Todo lo
que afirma procede del código del repo (`src/skillgen/training/
unsloth_backend.py`, testeado con un stub) y de la receta §18 de
PLAN.md. Los resultados reales de E1 existirán cuando este notebook
corra en Kaggle.

**Qué hace Unsloth aquí y qué NO hace:** Unsloth optimiza el lado del
modelo — carga 4-bit NF4, kernels de atención parcheados,
checkpointing de gradiente propio (~30% menos VRAM) — y acelera el
entrenamiento ~2× en la práctica. **No** sustituye nuestra tubería de
datos: el dataset lo construye `prepare.py` con labels assistant-only
**certificados** (ver la celda del gate §21). El entrenamiento corre
con `SFTTrainer` de TRL, que es la receta canónica de Unsloth:
Unsloth pone el modelo; TRL pone el loop.

## Celda (a) — Instalación (solo Kaggle)

`unsloth` **no** es dependencia del proyecto en local a propósito:
fija su propia pila de torch/xformers CUDA, que pelearía con el pin
de torch-CPU del desarrollo local, y bitsandbytes ni siquiera importa
sin GPU. En Kaggle la pila CUDA ya es la de la plataforma, así que
aquí sí se instala. Si pip cambia la versión de torch, Kaggle pedirá
reiniciar el kernel: hazlo y continúa desde la celda (b).

In [ ]:
# SOLO KAGGLE: en local estas instalaciones están prohibidas por diseño.
!pip install -q unsloth
!pip install -q trl peft accelerate datasets pyyaml
print("Instalación completa. Si unsloth cambió torch, reinicia el kernel y sigue desde la celda (b).")

## Celda (b) — `import unsloth` PRIMERO, siempre

Unsloth **parchea** `transformers`, `trl` y `peft` al importarse
(kernels de atención, checkpointing, internos del Trainer). Si
importas esas librerías antes, los parches no se aplican y pierdes
la aceleración en silencio. Por eso en TODO el proyecto la regla es:
**unsloth primero**. El módulo del repo
`skillgen/training/unsloth_backend.py` existe para encapsular esto:
importarlo importa unsloth (con guarda tipada si falta), y
`train.py` lo importa antes que torch/transformers/trl cuando el
backend es `unsloth`.

In [ ]:
import unsloth  # PRIMERO — parchea transformers/trl/peft. No muevas este import.
print("unsloth:", unsloth.__version__ if hasattr(unsloth, "__version__") else "ok")

from unsloth import FastLanguageModel
print("FastLanguageModel listo:", FastLanguageModel)

## Celda (c) — Verificación del entorno

Los mismos asserts del notebook `00_environment.ipynb`: sin GPU no se
entrena (PLAN §16); los kernels 4-bit exigen compute capability
≥ 7.0 (la T4 de Kaggle es 7.5; nada de Pascal/P100). Con VRAM < 16 GB
aplica la regla de reserva (§16): `max_seq_length` 2048, batch 1,
más acumulación — y no uses la config E1 tal cual.

In [ ]:
import sys

assert sys.version_info >= (3, 12), "SkillGen requiere Python >= 3.12"

import torch

print("torch:", torch.__version__)
assert torch.cuda.is_available(), "Sin GPU no se entrena (§16): activa la T4 en Settings → Accelerator."

props = torch.cuda.get_device_properties(0)
cc = props.major + props.minor / 10
vram_gb = props.total_memory / (1024 ** 3)
print(f"GPU: {props.name} | compute capability: {cc} | VRAM: {vram_gb:.1f} GB")

# Los kernels 4-bit de bitsandbytes/Unsloth exigen sm_70+ (nada de P100/Pascal).
assert cc >= 7.0, f"Compute capability {cc} < 7.0: cambia a una T4."
if vram_gb < 16:
    print("WARN: VRAM < 16 GB → aplica la regla de reserva de §16 (max_seq_length 2048, batch 1, más acumulación) y NO uses configs/qlora_e1.yaml tal cual.")
else:
    print("OK: entorno listo para E1 (max_seq_length 8192).")

## Celda (d) — Carga del modelo: `FastLanguageModel.from_pretrained`

Todos los valores salen de `configs/qlora_e1.yaml` (nada hardcodeado):

- `model_name` = `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`: el checkpoint
  **pre-cuantizado** de Unsloth (PLAN §15). El modelo base sigue
  siendo **PROVISIONAL** (Qwen2.5-7B-Instruct, pendiente de la
  verificación de modelos 2026).
- `load_in_4bit=True`: el base se carga en **4-bit NF4** con doble
  cuantización — ~3,5 GB en vez de ~14 GB en fp16. Los adaptadores
  LoRA se entrenan en fp16; por eso QLoRA pierde tan poco frente a
  LoRA completo.
- `max_seq_length=8192`: la decisión de la Fase 0 (medida con el EDA:
  el 46,75% del train supera 4096 tokens; se filtra, nunca se trunca).
- `dtype=None`: Unsloth elige solo (fp16 en la T4, que no tiene bf16).

En el repo, esta celda y la siguiente son exactamente lo que hace
`skillgen.training.unsloth_backend.load_unsloth_model(config)`.

In [ ]:
import sys
from pathlib import Path

# El repo debe estar en /kaggle/working (Dataset o clon). Ajusta si lo subiste a otra ruta.
REPO = Path("/kaggle/working/skillgen")
sys.path.insert(0, str(REPO / "src"))

from skillgen.training.config import load_config

config = load_config(REPO / "configs" / "qlora_e1.yaml")
print("Experimento:", config["experiment_id"], "| backend:", config["training"]["backend"])
assert config["training"]["backend"] == "unsloth"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config["model"]["name"],          # unsloth/Qwen2.5-7B-Instruct-bnb-4bit
    max_seq_length=config["data"]["max_seq_length"],  # 8192 (decisión Fase 0)
    dtype=None,                                  # auto: fp16 en T4
    load_in_4bit=True,                           # NF4 + doble cuantización
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Modelo cargado en 4-bit:", config["model"]["name"])

## Celda (e) — LoRA: `FastLanguageModel.get_peft_model`

Aquí se adjuntan los adaptadores LoRA (receta §18): **r=16, α=32,
dropout 0, bias none, todos los módulos lineales** (q/k/v/o/gate/up/
down) y la pieza clave de Unsloth:
`use_gradient_checkpointing="unsloth"` — su checkpointing propio,
que ahorra ~30% de VRAM frente al estándar.

`model.print_trainable_parameters()` debe mostrarte del orden de
**40,4 millones de parámetros entrenables (~0,5% del total)**: eso es
todo lo que se entrena. El resto del 7B queda congelado en 4-bit.
Cómo leerlo en entrevista: "entreno el 0,5% de los parámetros; el
adaptador resultante pesa ~100 MB y el base nunca se modifica".

In [ ]:
lora_cfg = config["lora"]
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],                                   # 16
    lora_alpha=lora_cfg["alpha"],                      # 32
    lora_dropout=lora_cfg["dropout"],                  # 0.0
    bias=lora_cfg["bias"],                             # "none"
    target_modules=lora_cfg["target_modules"],         # todos los lineales
    use_gradient_checkpointing="unsloth",              # ~30% menos VRAM
    random_state=config["training"]["seed"],           # 42
)
model.print_trainable_parameters()  # espera ≈40,4M entrenables (~0,5%)
model.config.use_cache = False      # incompatible con gradient checkpointing

## Celda (f) — Dataset y 🚦 GATE de masking (§21): PARA Y LEE

El dataset lo construye `prepare.py` del repo: renderiza con el chat
template nativo, **filtra** los registros que exceden 8192 tokens
(800 → 582) y construye los labels con **masking assistant-only**
(-100 en system/user; loss solo sobre el blueprint).

**Por qué NO usamos `train_on_responses_only` de Unsloth:** es una
alternativa válida y más corta, pero nuestro masking está
*certificado*: hay tests unitarios que decodifican los labels y
exigen que el texto visible sea exactamente el blueprint + EOS.
Cambiar a la utilidad de Unsloth sería cambiar una garantía probada
por una comodidad. Si algún día se migra, la migración correcta es
hacer pasar a `train_on_responses_only` por la misma verificación.

**El gate:** antes de entrenar, decodificamos los labels ≠ -100 del
primer ejemplo y lo imprimimos. Debe salir **SOLO el blueprint +
`<|im_end|>`**. Si saliera también la instrucción, o nada, el
template estaría mal aplicado y entrenarías basura con `loss`
engañosamente baja. Este gate es el mismo que ejecuta `train.py`.

In [ ]:
from skillgen.training.prepare import load_training_dataset, verify_masking

train_dataset, val_dataset, stats, first_record = load_training_dataset(tokenizer, config)
print("Dataset:", stats)

first_example = {"input_ids": train_dataset[0]["input_ids"], "labels": train_dataset[0]["labels"]}
decoded = verify_masking(first_example, tokenizer, first_record)  # lanza si el masking falla
print("=== TEXTO DESENMASCARADO (debe ser SOLO el blueprint + <|im_end|>) ===")
print(decoded)
print("=== FIN DEL GATE §21 ===")

## Celda (g) — Entrenamiento con SFTTrainer (la receta canónica de Unsloth)

Unsloth + `SFTTrainer` no es una mezcla rara: es **la** receta
oficial de Unsloth. Unsloth acelera el modelo (kernels parcheados,
checkpointing propio); TRL aporta el loop, el scheduler y el
checkpointing por mejor val loss.

Detalles que importan:

- **Sin `peft_config`:** el modelo YA trae LoRA aplicado desde la
  celda (e); pasarlo otra vez lo aplicaría doble.
- **Sin `gradient_checkpointing` en el Trainer:** ya está activado el
  de Unsloth en la celda (e); activar el estándar encima pelearía
  con él.
- Hiperparámetros §18 desde la config: lr 2e-4, 2 epochs, batch
  efectivo 16 (2 × acumulación 8), warmup 3%, cosine, `adamw_8bit`,
  seed 42, early stopping sobre val loss. **La regla de parada es el
  mejor checkpoint por val loss, no el final de las epochs.**
- Loss sana al final: **0,5–1,0**. Menos de ~0,2 huele a
  memorización; val subiendo con train bajando = overfitting.
- `report_to` queda vacío sin `WANDB_API_KEY` en el entorno (offline-
  safe); con la key, el run se registra en W&B (§28).

In [ ]:
import os

from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

t = config["training"]
output_dir = str(REPO / config["output_dir"])

sft_config = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=t["per_device_train_batch_size"],
    per_device_eval_batch_size=t["per_device_eval_batch_size"],
    gradient_accumulation_steps=t["gradient_accumulation_steps"],
    learning_rate=t["learning_rate"],
    num_train_epochs=t["num_train_epochs"],
    lr_scheduler_type=t["lr_scheduler_type"],
    warmup_steps=t["warmup_ratio"],  # transformers 5.x: float = ratio de pasos
    weight_decay=t["weight_decay"],
    optim=t["optim"],                # adamw_8bit (bitsandbytes, CUDA)
    gradient_checkpointing=False,    # ya activo el de Unsloth (celda e)
    max_grad_norm=t["max_grad_norm"],
    logging_steps=t["logging_steps"],
    eval_strategy="steps",
    eval_steps=t["eval_steps"],
    save_strategy="steps",
    save_steps=t["save_steps"],
    save_total_limit=t["save_total_limit"],
    load_best_model_at_end=True,     # mejor checkpoint por val loss
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=t["seed"],
    data_seed=t["seed"],
    report_to=["wandb"] if os.environ.get("WANDB_API_KEY") else [],
    max_length=config["data"]["max_seq_length"],
    packing=False,
    completion_only_loss=False,      # el masking es NUESTRO (prepare.py)
    assistant_only_loss=False,
    bf16=False,
    fp16=True,                       # T4: fp16 (sin bf16)
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=t["early_stopping_patience"])],
    # SIN peft_config: LoRA ya está aplicado (celda e).
)

train_result = trainer.train()
print("train loss:", train_result.training_loss)

## Celda (h) — Evaluación y rescate del adaptador

Entrenar no basta: evalúa y **rescata el adaptador antes de que la
sesión de Kaggle muera** (las sesiones gratuitas se cortan). Se
guarda SOLO el adaptador (~100 MB) + el tokenizer — el base se vuelve
a descargar cuando haga falta; guardar el modelo completo sería
gastar 14 GB para nada. Después, la evaluación de verdad (SkillGen
Score contra E0) se hace con el pipeline de la Fase 2/4 sobre
val/test, no con la val loss sola.

In [ ]:
eval_result = trainer.evaluate()
print("eval loss (val):", eval_result.get("eval_loss"))

trainer.save_model(output_dir)          # mejor checkpoint (por val loss)
model.save_pretrained(output_dir)       # adaptador LoRA (~100 MB)
tokenizer.save_pretrained(output_dir)
print("Adaptador + tokenizer en:", output_dir)

# Rescate: comprime y deja el zip en /kaggle/working para descargarlo.
!cd /kaggle/working/skillgen/experiments && zip -r /kaggle/working/e1_adapter.zip e1
print("Descarga e1_adapter.zip desde el panel Output de Kaggle ANTES de cerrar la sesión.")

## Celda (i) — Control rápido de inferencia: `FastLanguageModel.for_inference`

Antes de dar el run por bueno, una generación de control con una
instrucción de validación: `for_inference` activa la generación
rápida nativa de Unsloth. Lo esperado tras un E1 sano: un JSON que al
menos **parsea** y trae los campos del Blueprint (eso es lo que el
smoke sin entrenar NO lograba: inventaba el esquema). La puntuación
formal va después, con el harness, sobre el split completo.

In [ ]:
from skillgen.evaluation.baseline import load_split_records

model_inf = FastLanguageModel.for_inference(model)
val_record = load_split_records("val", limit=1, splits_dir=REPO / "data" / "splits")[0]

messages = [
    {"role": "system", "content": "You are SkillGen, an expert designer of reusable Agent Skills. Given the user request, output ONLY a valid JSON Skill Blueprint. No explanations, no markdown fences."},
    {"role": "user", "content": val_record["instruction"]},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output_ids = model_inf.generate(**inputs, max_new_tokens=1024, temperature=0.3, top_p=0.9)
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)[:2000])

## Celda (j) — (Opcional) Export GGUF: `save_pretrained_gguf`

El camino de export del proyecto (PLAN §44) es: adaptador → **merge
en 16-bit** → GGUF `q4_k_m` **desde el modelo fusionado** → Ollama.
Unsloth lo hace en una llamada con
`model.save_pretrained_gguf(dir, tokenizer,
quantization_method="q4_k_m")` — por debajo, fusiona primero en
16-bit y cuantiza después.

⚠️ **Nunca fusionar directamente a 4-bit:** perderías la precisión
del merge y el GGUF saldría degradado. El camino canónico y auditado
del repo es `scripts/export_model.py` (plan explícito, testeado);
esta celda es el atajo equivalente de Unsloth para una demo rápida
en la misma sesión.

In [ ]:
# OPCIONAL — demo rápida en sesión. El camino canónico del repo: scripts/export_model.py
# model.save_pretrained_gguf(str(REPO / "experiments/e1_gguf"), tokenizer, quantization_method="q4_k_m")
print("Celda opcional: descomenta la línea anterior para generar el GGUF q4_k_m en esta sesión.")

## Alternativa: el CLI del repo (mismo motor, sin notebook)

Todo lo anterior es exactamente lo que ejecuta el entry point del
repo cuando la config dice `backend: unsloth` — el notebook existe
para APRENDER la API paso a paso; el CLI existe para repetir runs
(E2–E6) sin tocar celdas:

```bash
cd /kaggle/working/skillgen
PYTHONPATH=src python -m skillgen.training.train --config configs/qlora_e1.yaml
```

## Qué sigue después de E1

- **Comparación pareada E0 vs E1** (Fase 4): mismo test congelado,
  McNemar + IC 95%, y la puerta §39: promover solo si el SkillGen
  Score sube ≥ +2 puntos sin regresión en el set experto.
- **Ablaciones (§19), un cambio por run**, cada una con su config ya
  creada en `configs/` (E2 masking, E3 epochs, E4 rango, E5 módulos,
  E6 tamaño de datos) — todas con `backend: unsloth`.
- **Export §44** con `scripts/export_model.py` e inspección del
  Modelfile de Ollama (el chat template debe ser el de entrenamiento).